# Планирование остановов станков на обслуживание

**Постановка.** На производственной линии работают N станков. Каждый станок работает циклами из трёх этапов. Каждый этап заканчивается плановым остановом на обслуживание:

| Этап | Вид обслуживания | Длительность останова |
|---|---|---|
| 1 | ТО-1 | 1 сут |
| 2 | ТО-2 | 2 сут |
| 3 | ТР (текущий ремонт) | 4 сут |

Перед каждым остановом станок должен отработать плановую наработку (около 30 сут). Наработку можно сократить или увеличить в пределах допуска (±15 %).

Требования к расписанию:
1. Фактическая наработка должна быть как можно ближе к плановой.
2. **Соседние станки не должны стоять на обслуживании одновременно:** иначе линия теряет два соседних участка сразу.
3. Число одновременных остановов не должно превышать число ремонтных бригад R. Загрузка бригад должна быть равномерной.

Задача решается пятью методами: MILP, CP-SAT, жадная эвристика H1, локальный поиск H2 и генетический алгоритм.

In [ ]:
# !pip install -r requirements.txt
import pandas as pd
from IPython.display import Image, display

from msched import Instance, generate, evaluate, solve
from msched.plots import plan_starts, schedule_figure
from msched.report import schedule_table, export_excel

pd.set_option("display.max_rows", 20)

## Пример задачи

Генератор создаёт пример: 10 станков в линию (соседи — станки i и i+1), горизонт планирования 180 сут, 3 бригады. Половина станков начинает с одинакового состояния, поэтому их плановые остановы часто совпадают.

In [ ]:
inst = generate(n_machines=10, horizon=180, seed=1, name="M_s1")
print(f"Станков: {inst.n_machines}, остановов: {inst.n}, бригад R = {inst.capacity}")
print(f"Пар соседних станков: {len(inst.neighbors)}, пар остановов, которые могут пересечься: {len(inst.pa)}")
schedule_table(inst, plan_starts(inst)).head(10)

## Математическая модель

Обозначения: $a$ — останов, $r_a$ — плановая наработка, $d_a$ — длительность, $[lo_a, hi_a]$ — допустимая наработка, $p(a)$ — предыдущий останов того же станка, $E$ — пары остановов соседних станков, $R$ — число бригад.

Переменные: $s_a$ — момент начала останова; $z_{ab} \in \{0,1\}$ — остановы $a$ и $b$ пересекаются; $P$ — пиковое число одновременных остановов.

Фактическая наработка: $L_a = s_a - (s_{p(a)} + d_{p(a)})$, для первого останова станка $L_a = s_a$.

$$\min F = w_{dev} \sum_a |L_a - r_a| + w_{conf} \sum_{(a,b) \in E} z_{ab} + w_{peak} P$$

при ограничениях:

$$lo_a \le L_a \le hi_a, \qquad \#\{a : s_a \le t < s_a + d_a\} \le P \le R \quad \forall t,$$

$$z_{ab} = 0 \;\Rightarrow\; s_a + d_a + \delta \le s_b \;\lor\; s_b + d_b + \delta \le s_a.$$

Веса по умолчанию: $w_{dev} = 1$, $w_{conf} = 20$, $w_{peak} = 10$.

In [ ]:
weights = inst.weights
print("Веса ЦФ:", weights)
plan = plan_starts(inst)
evaluate(inst, plan)

Плановое расписание (все остановы точно по плановой наработке) нарушает требования: много конфликтов соседей, а пик загрузки превышает число бригад. Задача — сдвинуть остановы в пределах допуска.

## 1. Модель MILP (PuLP)

Индексация по времени: бинарная переменная $x_{a,t} = 1$, если останов $a$ начинается в момент $t$. Модель строится в PuLP (`msched/milp.py`) и решается HiGHS или CBC.

In [ ]:
res_milp = solve(inst, "milp", time_limit=30)
print(res_milp.status, res_milp.info)
evaluate(inst, res_milp.starts)

## 2. Модель CP-SAT (OR-Tools)

Те же переменные $s_a$, но пересечения записаны логическими условиями (`only_enforce_if`), а загрузка бригад — глобальным ограничением `add_cumulative` (`msched/cpsat.py`).

In [ ]:
res_cpsat = solve(inst, "cpsat", time_limit=30)
print(res_cpsat.status, "нижняя оценка:", res_cpsat.bound)
evaluate(inst, res_cpsat.starts)

## 3. Эвристика H1 (жадная)

Остановы рассматриваются в порядке плановых моментов. Каждый останов ставится в момент из допустимого окна с минимальным приростом ЦФ относительно уже поставленных остановов.

In [ ]:
res_h1 = solve(inst, "h1")
evaluate(inst, res_h1.starts)

## 4. Эвристика H2 (итерированный локальный поиск)

H2 улучшает расписание H1. Ходы: сдвиг одного останова (`shift`) или сдвиг останова вместе со всеми следующими (`push`). В локальном минимуме расписание случайно возмущается, и спуск повторяется.

In [ ]:
res_h1h2 = solve(inst, "h1h2", time_limit=10)
evaluate(inst, res_h1h2.starts)

## 5. Генетический алгоритм

Хромосома — вектор наработок $L_a$. Кроссовер берёт гены станка целиком от одного из родителей, мутация сдвигает наработку на 1–3 сут.

In [ ]:
res_ga = solve(inst, "ga", time_limit=10)
print("Поколений:", res_ga.info["generations"])
evaluate(inst, res_ga.starts)

## 6. Сравнение методов на этом примере

In [ ]:
results = {"MILP": res_milp, "CP-SAT": res_cpsat, "H1": res_h1, "H1+H2": res_h1h2, "ГА": res_ga}
rows = []
for name, r in results.items():
    m = evaluate(inst, r.starts)
    rows.append({"Метод": name, "Статус": r.status, "ЦФ": m.objective, "Σ откл.": m.sum_dev,
                 "Конфликты": m.conflicts, "Пик": m.peak, "СКО загрузки": round(m.load_std, 3),
                 "До 1-го, с": round(r.t_first, 3), "До лучшего, с": round(r.t_best, 3),
                 "Всего, с": round(r.time_total, 2)})
pd.DataFrame(rows).set_index("Метод")

### Диаграмма Ганта: план и лучшее расписание

In [ ]:
schedule_figure(inst, {"План без сдвигов": plan, "CP-SAT": res_cpsat.starts}, "figures/demo_gantt.png")
display(Image("figures/demo_gantt.png"))

### Выгрузка расписания в Excel

In [ ]:
export_excel(inst, res_cpsat.starts, "results/schedule_M_s1.xlsx", method="CP-SAT")
schedule_table(inst, res_cpsat.starts).head(10)

## 7. Результаты вычислительных экспериментов

Полный прогон запускается из командной строки (около 70 мин):

```
bash scripts/run_experiments.sh
python -m msched plots
```

Ниже — средние значения по примерам размеров S (5 станков), M (10 станков) и L (20 станков), лимит времени 60 с на метод.

In [ ]:
summary = pd.read_csv("results/bench_summary.csv", index_col=[0, 1])
summary

In [ ]:
for name in ["comparison_gap.png", "comparison_time.png", "convergence_L_s1.png"]:
    display(Image(f"figures/{name}"))

### Анализ чувствительности (примеры M, лимит 20 с)

In [ ]:
for name in ["sens_tol.png", "sens_sync.png", "sens_capacity.png"]:
    display(Image(f"figures/{name}"))